In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow import keras
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

In [ ]:
# Step 1: Load and understand MNIST
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

print(f"Training images: {X_train.shape}")
print(f"Test images:     {X_test.shape}")
print(f"Number of classes: {len(np.unique(y_train))}")
print(f"Image dimensions: {X_train.shape[1:]}")

In [ ]:
# Display sample handwritten digits
fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for ax, img, label in zip(axes.flatten(), X_train[:10], y_train[:10]):
    ax.imshow(img, cmap="gray")
    ax.set_title(f"Label: {label}")
    ax.axis("off")

plt.suptitle("Sample MNIST Digits")
plt.tight_layout()
plt.show()

In [ ]:
# Inspect the pixel values of one image
print("Pixel array of the first image:")
print(X_train[0][:5, :5])
print(f"Minimum pixel value: {X_train.min()}")
print(f"Maximum pixel value: {X_train.max()}")

In [ ]:
# Step 2: Preprocess the data
# Preserve original test images for later visualisation and prediction
X_test_raw = X_test.copy()

# Normalise pixel values from 0-255 to 0-1
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# Flatten images from 28x28 grids into 784 features
X_train_flat = X_train.reshape(-1, 784)
X_test_flat = X_test.reshape(-1, 784)

# One-hot encode the digit labels
y_train_ohe = keras.utils.to_categorical(y_train, 10)
y_test_ohe = keras.utils.to_categorical(y_test, 10)

print(f"Flattened training data: {X_train_flat.shape}")
print(f"Flattened test data: {X_test_flat.shape}")
print(f"One-hot label for {y_train[0]}: {y_train_ohe[0]}")

In [ ]:
# Step 3: Build the neural network
model = keras.Sequential([
    keras.layers.Input(shape=(784,)),
    keras.layers.Dense(256, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(10, activation="softmax")
])

model.summary()

In [ ]:
# Step 4: Compile the model
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Stop training when validation loss stops improving
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

In [ ]:
# Train the model using 10% of training data for validation
history = model.fit(
    X_train_flat,
    y_train_ohe,
    epochs=20,
    batch_size=128,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Step 5: Visualise training and validation performance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["accuracy"], label="Training")
axes[0].plot(history.history["val_accuracy"], label="Validation")
axes[0].set_title("Training vs Validation Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(history.history["loss"], label="Training")
axes[1].plot(history.history["val_loss"], label="Validation")
axes[1].set_title("Training vs Validation Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Step 6: Evaluate on the unseen test dataset
test_loss, test_acc = model.evaluate(
    X_test_flat,
    y_test_ohe,
    verbose=0
)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.1%}")

In [ ]:
# Step 7: Generate predictions
y_pred = model.predict(X_test_flat, verbose=0)

# Select the digit with the highest predicted probability
y_pred_classes = y_pred.argmax(axis=1)

# Find incorrectly classified images
wrong_idx = np.where(y_pred_classes != y_test)[0]

print(f"Wrong predictions: {len(wrong_idx)} out of {len(y_test)}")

# Display up to ten incorrect predictions
if len(wrong_idx) > 0:
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))

    for ax, idx in zip(axes.flatten(), wrong_idx[:10]):
        ax.imshow(X_test_raw[idx], cmap="gray")
        ax.set_title(
            f"True: {y_test[idx]} | Pred: {y_pred_classes[idx]}",
            color="red"
        )
        ax.axis("off")

    plt.suptitle("What the Model Gets Wrong")
    plt.tight_layout()
    plt.show()
else:
    print("No incorrect predictions found.")

In [ ]:
# Step 8: Evaluate individual classes using a confusion matrix
cm = confusion_matrix(y_test, y_pred_classes)

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=np.arange(10)
)

display.plot(cmap="Blues", values_format="d")
plt.title("MNIST Confusion Matrix")
plt.show()

In [ ]:
# Display precision, recall and F1-score for each digit
print("Classification Report:")
print(classification_report(y_test, y_pred_classes, zero_division=0))

In [ ]:
# Step 9: Create a reusable prediction function
def predict_digit(image):
    flat = image.reshape(1, 784).astype("float32") / 255.0

    probabilities = model.predict(flat, verbose=0)[0]

    predicted_digit = probabilities.argmax()
    confidence = probabilities.max()

    return predicted_digit, confidence

# Test the prediction function on three unseen images
for i in range(3):
    digit, confidence = predict_digit(X_test_raw[i])

    print(
        f"True: {y_test[i]} | "
        f"Predicted: {digit} | "
        f"Confidence: {confidence:.1%}"
    )

    plt.imshow(X_test_raw[i], cmap="gray")
    plt.title(f"Predicted: {digit} | Confidence: {confidence:.1%}")
    plt.axis("off")
    plt.show()